# cis100 correct-epoch analogue selection + powderday RT staging

The existing cis100 powderday sample (PSB forward-modeling selection) has no completed RT above
z = 1.145 and its members are too young. This notebook does the **correct selection directly from
the full m100n1024 population at the right snapshots** and stages the powderday runs; you submit
the printed sbatch commands, then the analysis notebook (`analogues_specphot_cis100.ipynb`) can be
pointed at the new tree.

Anchors (nearly exact epoch matches):
- **000719** (z = 1.463) → snap **091** (z = 1.4584)
- **002962** (z = 1.266) → snap **096** (z = 1.2780)

Pre-computed funnel (mass+age windows, sSFR < 0.2/t_H, ngas ≥ 21 & nstar ≥ 20):
719 → **156** matches (26 AGN / 130 non-AGN); 2962 (±0.4 dex, ±1.0 Gyr) → **72** (13 AGN / 59).
With the default cap of 40 non-AGN per target (nearest matches; all AGN kept) that is
**~119 RT runs** total. Set `MAX_PER_GROUP = None` to stage all 228.

Pipeline = same as `test_powderday.ipynb`: caesar-based selection → `extract_particles`
(PartType0+4, prefix `m100n1024`, shared `filtered_particles/` tree, existing files skipped) →
`MakeSED.selection_gals` + `create_master('cluster', 'plist')` → sbatch.
New run tree: `output/cis100/sed_analogues/dusty_simdust/` (dust-on, same `parameters_master.py`
as the previous cis100 run: single sightline, kmh94, live SIMBA dust).


In [ ]:
# ── Part 0 · config ──────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.cosmology import Planck13
import astropy.units as u
from astropy import constants as const

from simbanator.io.simba import Simulation
from simbanator.sed.makesed import MakeSED

sim = Simulation('cis100')

HOME            = '/mnt/home/glorenzon/analize_simba_cgm'
OUTDIR          = os.path.join(HOME, 'output', 'cis100', 'analogues_specphot')     # selection tables
SED_OUT         = os.path.join(HOME, 'output', 'cis100', 'sed_analogues')          # new RT tree
RUN_TAG         = 'dusty_simdust'
PARAMF          = 'parameters_master.py'            # dust-on master (as sed_dust_comparison)
SELECTION_FILE  = 'selection_analogues_specphot.hdf5'
PARTICLE_PREFIX = 'm100n1024'
hydro_dir_base  = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')  # shared tree
PARTITION       = 'INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL'
os.makedirs(OUTDIR, exist_ok=True)

TARGETS = {
    '719':  dict(z=1.463, snap=91, z_snap=1.4584, logm=10.54, age_gyr=2.35,
                 dlogm=0.30, dage_gyr=0.75),
    '2962': dict(z=1.266, snap=96, z_snap=1.2780, logm=10.92, age_gyr=3.30,
                 dlogm=0.40, dage_gyr=1.00),
}

PASSIVE_FACTOR = 0.2       # quiescent: sSFR < PASSIVE_FACTOR / t_H(z)
NGAS_MIN, NSTAR_MIN = 21, 20
EPS_RAD, AGN_FEDD_MIN, AGN_LBOL_MIN = 0.1, 0.02, 1e43

# cap per (target, AGN-class) group, keeping the NEAREST matches in normalized (mass, age)
# distance; AGN groups are below the cap anyway, so this only trims the non-AGN bulk.
MAX_PER_GROUP = 40         # None -> stage everything (228 RT runs)
print('selection tables ->', OUTDIR, '\nRT tree ->', os.path.join(SED_OUT, RUN_TAG))


In [ ]:
# ── Part 1 · correct-epoch selection from the caesar catalogs ────────────────
SELECTED = {}
for tid, T in TARGETS.items():
    with h5py.File(sim.get_caesar_file(T['snap']), 'r') as f:
        d = f['galaxy_data']
        gid  = np.asarray(d['GroupID'][:], int)
        ms   = np.asarray(d['dicts/masses.stellar'][:], float)
        age  = np.asarray(d['dicts/ages.mass_weighted'][:], float)
        md   = np.asarray(d['dicts/masses.dust'][:], float)
        mgas = np.asarray(d['dicts/masses.gas'][:], float)
        sfr  = np.asarray(d['sfr'][:], float)
        ngas = np.asarray(d['ngas'][:], int)
        nstar= np.asarray(d['nstar'][:], int)
        cen  = np.asarray(d['central'][:], int)
        fedd = np.asarray(d['bh_fedd'][:], float)
        mdot = np.asarray(d['bhmdot'][:], float)
        r50s = np.asarray(d['dicts/radii.stellar_half_mass'][:], float)

    with np.errstate(invalid='ignore', divide='ignore'):
        lm   = np.log10(np.where(ms > 0, ms, np.nan))
        ssfr = sfr / ms
        lbol = (EPS_RAD * (mdot * u.Msun / u.yr) * const.c**2).to(u.erg / u.s).value
    agn = (np.nan_to_num(fedd) >= AGN_FEDD_MIN) & (np.nan_to_num(lbol) >= AGN_LBOL_MIN)
    tH  = Planck13.age(T['z_snap']).to(u.yr).value

    m_m = np.abs(lm - T['logm']) <= T['dlogm']
    m_a = np.abs(age - T['age_gyr']) <= T['dage_gyr']
    m_q = ssfr < PASSIVE_FACTOR / tH
    m_r = (ngas >= NGAS_MIN) & (nstar >= NSTAR_MIN)
    sel = m_m & m_a & m_q & m_r
    print(f"── {tid}  snap {T['snap']} (z={T['z_snap']}) ──")
    print(f"  mass window          : {int(m_m.sum()):5d}")
    print(f"  + age window         : {int((m_m & m_a).sum()):5d}")
    print(f"  + quiescent          : {int((m_m & m_a & m_q).sum()):5d}")
    print(f"  + resolution floors  : {int(sel.sum()):5d}"
          f"   -> AGN {int((sel & agn).sum())} | non-AGN {int((sel & ~agn).sum())}")

    # nearest-match cap per AGN class (normalized by the window half-widths)
    dist = np.sqrt(((lm - T['logm']) / T['dlogm'])**2 + ((age - T['age_gyr']) / T['dage_gyr'])**2)
    keep = np.zeros(sel.size, bool)
    for cls_mask in (sel & agn, sel & ~agn):
        ii = np.where(cls_mask)[0]
        if MAX_PER_GROUP is not None and ii.size > MAX_PER_GROUP:
            ii = ii[np.argsort(dist[ii])[:MAX_PER_GROUP]]
        keep[ii] = True
    print(f"  staged (cap {MAX_PER_GROUP}/group): {int(keep.sum())}"
          f"   -> AGN {int((keep & agn).sum())} | non-AGN {int((keep & ~agn).sum())}")

    A = Table(dict(SNAPSHOT=np.full(keep.sum(), T['snap']), GROUPID_SNAPSHOT=gid[keep],
                   REDSHIFT=np.full(keep.sum(), T['z_snap']),
                   LOG_MSTAR=lm[keep], AGE_STAR=age[keep], SSFR=ssfr[keep],
                   MGAS=mgas[keep], MDUST=md[keep],
                   LOG_MDUST=np.log10(np.where(md[keep] > 0, md[keep], np.nan)),
                   NGAS=ngas[keep], NSTAR=nstar[keep], CENTRAL=cen[keep],
                   FEDD=fedd[keep], LBOL=lbol[keep], IS_AGN=agn[keep],
                   R50_STAR=r50s[keep], DIST_MATCH=dist[keep]))
    A['AGN_CLASS'] = np.where(np.asarray(A['IS_AGN']), 'AGN', 'no_AGN')
    A.sort('GROUPID_SNAPSHOT')
    out = os.path.join(OUTDIR, f'rt_selection_{tid}.fits')
    A.write(out, overwrite=True)
    SELECTED[tid] = A
    with np.errstate(invalid='ignore'):
        print(f"  logMdust med {np.nanmedian(A['LOG_MDUST']):.2f}  -> {out}\n")

# combined unique (snap, gal) list for extraction + staging
_pairs = np.unique(np.concatenate([np.stack([np.asarray(A['SNAPSHOT'], int),
                                             np.asarray(A['GROUPID_SNAPSHOT'], int)], axis=1)
                                   for A in SELECTED.values()]), axis=0)
SNAPS, IDS = _pairs[:, 0], _pairs[:, 1]
print(f'total unique RT targets: {len(SNAPS)} over snaps {sorted(set(SNAPS.tolist()))}')


In [ ]:
# ── Part 2 · particle extraction (heavy: reads each ~265 GB snapshot once) ───
# Same as test_powderday Stage 0; existing files under filtered_particles/ are skipped.
from simbanator.analysis import extract_particles

EXTRACT_OVERWRITE = False
EXTRACT_PTYPES    = ("PartType0", "PartType4")   # gas (carries Dust_Masses) + stars

for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    _cs = sim.load_catalog(snap=_snap)
    extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, ptypes=EXTRACT_PTYPES,
                      sim_name=sim.name, prefix=PARTICLE_PREFIX,
                      overwrite=EXTRACT_OVERWRITE, verbose=1)
    del _cs
print('\nparticle extraction complete ->', hydro_dir_base)


In [ ]:
# ── Part 3 · stage the powderday run (jobs written, NOT submitted) ───────────
ms = MakeSED(sim, nnodes=1, model_run_name=RUN_TAG,
             hydro_dir_base=hydro_dir_base, selection_file=SELECTION_FILE,
             output_dir=SED_OUT, run_tag=RUN_TAG)
ms.selection_gals(snaps=SNAPS, galaxyID=IDS)
ms.create_master('cluster', 'plist', radius=None, partition=PARTITION,
                 prefix=PARTICLE_PREFIX, paramf=PARAMF, snaps_to_run=None)

# NOTE: never assume the job-file naming — the installed simbanator (pd39 site-packages) writes
# unpadded names (master.snap91.job) while newer repo code pads (master.snap091.job). Glob the
# directory for whatever was actually written.
import glob
print('\n── submit with ──')
for _snap in sorted(set(int(s) for s in SNAPS)):
    jdir = os.path.join(SED_OUT, RUN_TAG, 'powderday_sed_out', f'snap_{_snap:03d}')
    if not os.path.isdir(jdir):                       # unpadded dir fallback
        jdir = os.path.join(SED_OUT, RUN_TAG, 'powderday_sed_out', f'snap_{_snap}')
    jobs = sorted(glob.glob(os.path.join(jdir, 'master.snap*.job')))
    if jobs:
        print(f'cd {jdir} && sbatch {os.path.basename(jobs[-1])}')
    else:
        print(f'!! no master.snap*.job found in {jdir} — staging failed?')


## Part 4 · after the RT finishes

1. Check completeness: count `*.rtout.sed` under
   `output/cis100/sed_analogues/dusty_simdust/powderday_sed_out/snap_{091,096}/gal_*/`.
2. Analyze with `analogues_specphot_cis100.ipynb` by switching its inputs to this run:
   - `SED_ROOT = .../output/cis100/sed_analogues/dusty_simdust/powderday_sed_out`
   - sample = `analogues_specphot/rt_selection_{719,2962}.fits` (already carries
     `LOG_MSTAR/AGE_STAR/SSFR/LOG_MDUST/FEDD/LBOL/IS_AGN`, so Parts 3–4 of that notebook
     collapse to a table read; no BH harvest and no time-window matching needed — every
     analogue already sits at the correct epoch, mass, and age).
3. Expectation to test: matched analogues have median log M_dust ≈ 4.7 (719 window) and
   ≈ 5.5 (2962 window) vs CIGALE ≈ 6.7 (upper limit) / 7.2 — the optical should now agree
   (correct mass AND age) while the FIR shows the real dust discrepancy.

Caveats: dust-on only (add a `nodust_1e-12` clone of Part 3 if the dust-off comparison is
wanted); single sightline, no AGN emission source (same `parameters_master.py` as before,
for strict comparability with the existing 592-galaxy run).
